## Import Library

In [ ]:
import cv2
import numpy as np

## Points Reorder

In [ ]:
def reorder(points):
    points = points.reshape((4, 2))
    new_points = np.zeros((4, 1, 2), dtype=np.int32)

    add = points.sum(1)
    new_points[0] = points[np.argmin(add)]   # top-left
    new_points[3] = points[np.argmax(add)]   # bottom-right

    diff = np.diff(points, axis=1)
    new_points[1] = points[np.argmin(diff)]  # top-right
    new_points[2] = points[np.argmax(diff)]  # bottom-left

    return new_points

## Perspective Transform

In [ ]:
def warp(img, points, w, h):
    points = reorder(points)
    pts1 = np.float32(points)
    pts2 = np.float32([[0, 0], [w, 0], [0, h], [w, h]])

    matrix = cv2.getPerspectiveTransform(pts1, pts2)
    output = cv2.warpPerspective(img, matrix, (w, h))
    return output

In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = frame.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 1)
    edges = cv2.Canny(blur, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    biggest = None
    max_area = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area > 5000:
            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)

            if area > max_area and len(approx) == 4:
                biggest = approx
                max_area = area

    if biggest is not None:
        cv2.drawContours(img, biggest, -1, (0, 255, 0), 3)
        scanned = warp(frame, biggest, 500, 700)
        scanned_gray = cv2.cvtColor(scanned, cv2.COLOR_BGR2GRAY)
        scanned_thresh = cv2.adaptiveThreshold(
            scanned_gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 21, 3
        )
        cv2.imshow("Scanned", scanned_thresh)

    cv2.imshow("Original", img)
    cv2.imshow("Edges", edges)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()